## 3-1 Accuracy(정확도)

In [ ]:
# scikit-learn 라이브러리 import 및 버전 확인
# - 머신러닝 모델 학습/평가에 필요한 핵심 라이브러리
# - 버전에 따라 일부 API/파라미터가 달라질 수 있으므로 버전 체크는 좋은 습관
import sklearn

print(sklearn.__version__)

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator

# [원리] 정확도(Accuracy) 지표가 가진 한계를 보이기 위한 "더미 분류기"
# - BaseEstimator를 상속하면 사이킷런 호환 추정기(Estimator)가 되어
#   fit/predict 인터페이스만 맞추면 다른 클래스처럼 사용 가능
# - 이 분류기는 실제 학습을 수행하지 않고, 단순한 규칙(성별)만으로 예측
#   → 즉, 데이터가 한쪽 클래스로 치우쳐 있으면 단순 규칙만으로도
#     정확도가 높게 나올 수 있음을 보여주는 예시
class MyDummyClassifier(BaseEstimator):
    # fit( ) 메소드는 아무것도 학습하지 않음.
    # - pass만 두어 학습 단계를 비워둠 → 어떠한 파라미터/가중치도 갱신하지 않음
    def fit(self, X , y=None):
        pass
    
    # predict( ) 메소드는 단순히 Sex feature가 1 이면 0 , 그렇지 않으면 1 로 예측함.
    # [작동 방식]
    # 1) 입력 데이터 X의 행 개수만큼 0으로 채워진 (n,1) ndarray pred 생성
    # 2) 각 행을 순회하며 'Sex' 값이 1(남성)이면 사망(0), 0(여성)이면 생존(1)으로 지정
    #    → 타이타닉 도메인 지식: 여성/아이의 생존율이 높았다는 사실을 단순 규칙화
    # 3) 학습 없이 규칙만으로 예측을 반환
    def predict(self, X):
        pred = np.zeros( ( X.shape[0], 1 ))
        for i in range (X.shape[0]) :
            if X['Sex'].iloc[i] == 1:
                pred[i] = 0
            else :
                pred[i] = 1
        
        return pred


In [ ]:
from sklearn.preprocessing import LabelEncoder

# [원리] 머신러닝은 결측값/문자열을 그대로 받지 못함 → 전처리 필요
# 아래 함수들은 타이타닉 데이터셋 전처리를 단계별로 분리해 재사용성을 높임

# Null 처리 함수
# - Age: 평균값으로 대체(연속형이라 평균이 무난한 대체값)
# - Cabin: 'N'이라는 문자열로 채움(누락이 매우 많아 별도 카테고리로 처리)
# - Embarked: 'N'으로 채움(범주형 결측 → 새 카테고리로 처리)
# - Fare: 0으로 채움(요금 누락 시 0으로 가정)
# inplace=True : 원본 DF를 직접 수정 (새 객체 반환 X)
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# 머신러닝 알고리즘에 불필요한 피처 제거
# - PassengerId: 단순 식별자(예측에 무의미)
# - Name: 텍스트 자체는 직접적 패턴이 약함(이번 예제에선 사용 X)
# - Ticket: 표 번호도 패턴화하기 어려워 제거
# axis=1 : 컬럼 방향 삭제
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행.
# [원리] 문자열 카테고리 → 정수형 코드로 변환해야 모델 입력 가능
# - Cabin은 첫 글자만 사용(객실 등급 구간을 의미)해 카디널리티를 낮춤
# - LabelEncoder는 알파벳 순으로 0,1,2,...를 부여
#   주의: 트리 계열에는 무난하지만, 선형 모델에서는 순서 의미가 잘못 전달될 수 있음
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
# [작동 방식] 결측치 처리 → 불필요 컬럼 제거 → 범주형 인코딩 순서로 일관 적용
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 원본 데이터를 재로딩, 데이터 가공, 학습 데이터/테스트 데이터 분할.
# [작동 방식]
# 1) titanic_train.csv를 DataFrame으로 읽어옴
# 2) 'Survived'를 타깃(y), 나머지를 피처(X)로 분리
# 3) transform_features로 X에 대해 결측 처리/불필요 컬럼 제거/인코딩 수행
# 4) train_test_split으로 학습:테스트 = 80:20 으로 분할
#    - random_state=0 : 셔플 시드 고정 → 재현 가능
titanic_df = pd.read_csv('./titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df= titanic_df.drop('Survived', axis=1)
X_titanic_df = transform_features(X_titanic_df)
X_train, X_test, y_train, y_test=train_test_split(X_titanic_df, y_titanic_df,
                                                  test_size=0.2, random_state=0)

# 위에서 생성한 Dummy Classifier를 이용해 학습/예측/평가 수행.
# [원리]
# - fit은 아무 일도 하지 않고, predict는 단순 규칙(여성=생존)만 적용
# - 그럼에도 정확도가 약 78%로 높게 나옴
#   → 타이타닉 데이터의 성별-생존 상관이 강하기 때문
# - accuracy_score(y_true, y_pred) = 맞춘 개수 / 전체 개수
#   → 클래스 불균형/도메인 편향이 있을 때 성능을 과대평가할 수 있다는 교훈
myclf = MyDummyClassifier()
myclf.fit(X_train, y_train)

mypredictions = myclf.predict(X_test)
print('Dummy Classifier의 정확도는: {0:.4f}'.format(accuracy_score(y_test, mypredictions)))

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# [원리] "불균형 데이터에서 정확도 지표가 무력해지는" 현상을 시연하기 위한 분류기
# - 어떤 입력이든 무조건 0(False)로 예측
# - 양성 비율이 매우 낮은 데이터에선 "모두 0" 만으로도 정확도가 매우 높게 나옴
class MyFakeClassifier(BaseEstimator):
    def fit(self,X,y):
        pass
    
    # 입력값으로 들어오는 X 데이터 셋의 크기만큼 모두 0값으로 만들어서 반환
    # - shape: (len(X), 1), dtype=bool → 모두 False
    # - 즉, 어떤 샘플이든 "음성(Negative)"으로 예측
    def predict(self,X):
        return np.zeros( (len(X), 1) , dtype=bool)

# 사이킷런의 내장 데이터 셋인 load_digits( )를 이용하여 MNIST 데이터 로딩
# [원리]
# - load_digits: 8x8 손글씨 숫자(0~9) 이미지 약 1,797장
# - digits.data: (1797, 64) 형태의 픽셀 값 (이미지 1장을 64차원 벡터로 펼친 것)
# - digits.target: 각 이미지의 실제 숫자 라벨(0~9)
digits = load_digits()

print(digits.data)
print("### digits.data.shape:", digits.data.shape)
print(digits.target)
print("### digits.target.shape:", digits.target.shape)

In [ ]:
# [원리] 다중분류(0~9) 문제를 "이진분류(7인가? 아닌가?)"로 변환하기 위한 사전 준비
# - digits.target 배열의 각 원소를 7과 비교 → 같으면 True, 아니면 False
# - 결과는 같은 길이(1797)의 boolean ndarray
# - 양성(7)은 약 10%에 불과 → 강한 클래스 불균형이 만들어짐
digits.target == 7

In [ ]:
# digits번호가 7번이면 True이고 이를 astype(int)로 1로 변환, 7번이 아니면 False이고 0으로 변환. 
# [작동 방식]
# - boolean ndarray에 .astype(int)를 호출하면 True→1, False→0 으로 변환됨
# - 결과 y는 "7이면 1, 아니면 0"인 이진 라벨 ndarray
# - 그 다음 train_test_split으로 학습/테스트 분할
#   * random_state=11 : 결과 재현성을 위해 난수 시드 고정
#   * 기본 test_size=0.25 (75:25 분할)
y = (digits.target == 7).astype(int)
X_train, X_test, y_train, y_test = train_test_split( digits.data, y, random_state=11)

In [ ]:
# 불균형한 레이블 데이터 분포도 확인. 
# [원리/작동 방식]
# - y_test에서 0(7이 아님)과 1(7) 클래스의 빈도수를 확인
#   → 0:405, 1:45 → 양성 비율이 약 10%인 강한 불균형
# - 이렇게 불균형이 클 때 "모두 0으로 예측"만 해도 정확도가 405/450=0.9 이상 나옴
print('레이블 테스트 세트 크기 :', y_test.shape)
print('테스트 세트 레이블 0 과 1의 분포도')
print(pd.Series(y_test).value_counts())

# Dummy Classifier로 학습/예측/정확도 평가
# [핵심 메시지]
# - 어떤 학습도 하지 않고 무조건 0으로 예측해도 정확도 90% 달성
#   → 정확도(accuracy) 단일 지표는 불균형 분류 문제에서 모델 성능을 잘못 알려줄 수 있음
#   → 그래서 정밀도/재현율/F1/ROC-AUC 같은 보조 지표가 반드시 필요함
fakeclf = MyFakeClassifier()
fakeclf.fit(X_train , y_train)
fakepred = fakeclf.predict(X_test)
print('모든 예측을 0으로 하여도 정확도는:{:.3f}'.format(accuracy_score(y_test , fakepred)))

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# 앞절의 예측 결과인 fakepred와 실제 결과인 y_test의 Confusion Matrix출력
# [원리] Confusion Matrix(오차 행렬)
#   기본 형태(2x2):
#       [[TN, FP],
#        [FN, TP]]
#   - TN(True Negative)  : 실제 0 → 예측 0 (맞춤)
#   - FP(False Positive) : 실제 0 → 예측 1 (틀림, 1종 오류)
#   - FN(False Negative) : 실제 1 → 예측 0 (틀림, 2종 오류)
#   - TP(True Positive)  : 실제 1 → 예측 1 (맞춤)
#
# [작동 방식]
# - 모든 샘플을 0으로 예측 → Positive(1)로 예측한 샘플이 0개
#   → FP=0, TP=0
# - 결과: [[405, 0], [45, 0]]
#   * 실제 0인 405건은 모두 0으로 예측해서 맞춤(TN=405)
#   * 실제 1인 45건은 전부 0으로 잘못 예측(FN=45)
# → 양성 검출 능력이 0이라는 사실이 한눈에 드러남
confusion_matrix(y_test , fakepred)

## 정밀도(Precision) 과 재현율(Recall)

**MyFakeClassifier의 예측 결과로 정밀도와 재현율 측정**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score , recall_score

# [원리]
# - 정밀도(Precision) = TP / (TP + FP)
#   "Positive로 예측한 것 중 실제로 Positive인 비율"
#   → 예측의 정확함(허위 경보가 적을수록 좋음)
# - 재현율(Recall)    = TP / (TP + FN)
#   "실제 Positive 중 모델이 Positive로 잡아낸 비율"
#   → 놓치지 않는 능력(탐지율, Sensitivity)
#
# [작동 방식]
# - fakepred는 모두 0(음성) → TP=0, FP=0, FN=45
#   * Precision = 0/(0+0) → 분모가 0이라 "0으로 정의"되며 sklearn은 경고 출력
#   * Recall    = 0/(0+45) = 0
# → 정확도가 90%처럼 "좋아 보여도" 양성 탐지력이 0임을 정밀도/재현율이 폭로
print("정밀도:", precision_score(y_test, fakepred))
print("재현율:", recall_score(y_test, fakepred))

**오차행렬, 정확도, 정밀도, 재현율을 한꺼번에 계산하는 함수 생성**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score , recall_score , confusion_matrix

# [원리] 평가에 자주 쓰는 4가지 지표를 한 함수로 묶어 매번 호출하기 편하게 만든 유틸 함수
# - confusion_matrix : TN/FP/FN/TP 분포를 한눈에
# - accuracy        : 전체 정답률
# - precision       : 양성 예측의 신뢰도
# - recall          : 양성 검출 능력
#
# [작동 방식]
# - y_test와 모델이 만든 pred를 받아 위 4개 지표를 계산해 출력
# - 이후 셀들에서 다양한 임곗값/모델 결과를 비교할 때 반복 사용
def get_clf_eval(y_test , pred):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}'.format(accuracy , precision ,recall))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression

# 원본 데이터를 재로딩, 데이터 가공, 학습데이터/테스트 데이터 분할.
# [원리]
# - 로지스틱 회귀(Logistic Regression):
#   선형 조합 wx+b 를 시그모이드 함수로 [0,1] 확률에 매핑하는 이진 분류기
#   기본 임곗값 0.5 기준으로 0/1 클래스를 결정
#
# [작동 방식]
# 1) 데이터 로딩 → 전처리(transform_features) → 8:2로 분할
# 2) LogisticRegression(solver='liblinear')
#    - liblinear : 이진/소규모 데이터에 적합한 좌표 하강법 기반 솔버
# 3) fit(X_train, y_train) : 손실(로지스틱 로스) 최소화 방향으로 가중치 학습
# 4) predict(X_test)       : 임곗값 0.5 기준의 0/1 라벨 반환
# 5) get_clf_eval로 오차행렬/정확도/정밀도/재현율 동시 출력
titanic_df = pd.read_csv('./titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df= titanic_df.drop('Survived', axis=1)
X_titanic_df = transform_features(X_titanic_df)

X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, \
                                                    test_size=0.20, random_state=11)

lr_clf = LogisticRegression(solver='liblinear')

lr_clf.fit(X_train , y_train)
pred = lr_clf.predict(X_test)
get_clf_eval(y_test , pred)

### Precision/Recall Trade-off

**predict_proba( ) 메소드 확인**

In [ ]:
# [원리] 정밀도/재현율 트레이드오프를 이해하기 위해선 "예측 확률"이 필요
# - predict()  : 임곗값(기본 0.5) 적용 후의 최종 클래스 (0 또는 1)
# - predict_proba() : 각 클래스에 대한 예측 확률
#   * 반환 shape = (n_samples, n_classes)
#   * 컬럼 0 = P(class=0),  컬럼 1 = P(class=1)
#   * 두 컬럼의 합 = 1.0
#
# [작동 방식]
# - lr_clf는 학습된 모델 → 각 테스트 샘플에 대해 [P(0), P(1)] 확률을 산출
# - argmax(또는 0.5 임곗값) 기준으로 두 확률 중 큰 쪽이 최종 예측 클래스
# - 아래는 확률과 최종 예측을 가로로 붙여 한눈에 확인하기 위한 디버깅 코드
pred_proba = lr_clf.predict_proba(X_test)
pred  = lr_clf.predict(X_test)
print('pred_proba()결과 Shape : {0}'.format(pred_proba.shape))
print('pred_proba array에서 앞 3개만 샘플로 추출 \n:', pred_proba[:3])

# 예측 확률 array 와 예측 결과값 array 를 concatenate 하여 예측 확률과 결과값을 한눈에 확인
# - pred.reshape(-1,1) : 1차원 → (n,1) 2차원으로 변환해 axis=1로 concat 가능하게 만듦
# - 결과: [P(0), P(1), 최종 예측 라벨] 형태
pred_proba_result = np.concatenate([pred_proba , pred.reshape(-1,1)],axis=1)
print('두개의 class 중에서 더 큰 확률을 클래스 값으로 예측 \n',pred_proba_result[:3])


**Binarizer 활용**

In [ ]:
from sklearn.preprocessing import Binarizer

# [원리] Binarizer
# - threshold 값을 기준으로 각 원소를 0/1로 변환하는 전처리기
# - 규칙: 값이 threshold "이하"면 0, threshold "초과"면 1
#   (참고: <= 가 0, > 가 1)
# - 분류 결정 임곗값을 직접 바꿔보기 위해 사용 (predict_proba 결과에 적용)
X = [[ 1, -1,  2],
     [ 2,  0,  0],
     [ 0,  1.1, 1.2]]

# threshold 기준값보다 같거나 작으면 0을, 크면 1을 반환
# [작동 방식 예시] threshold=1.1
# - 1   <=1.1 → 0
# - -1  <=1.1 → 0
# - 2   > 1.1 → 1
# - 1.1 <=1.1 → 0  (등호 케이스: 0이 됨)
# - 1.2 > 1.1 → 1
binarizer = Binarizer(threshold=1.1)                     
print(binarizer.fit_transform(X))

**분류 결정 임계값 0.5 기반에서 Binarizer를 이용하여 예측값 변환**

In [ ]:
from sklearn.preprocessing import Binarizer

# [원리] 분류 결정 임곗값(Threshold)을 명시적으로 바꿔 예측을 재구성
# - 사이킷런 분류기의 predict()는 내부적으로 임곗값 0.5를 사용
# - 같은 결과를 직접 만들고 싶다면 predict_proba의 양성 확률 컬럼에 0.5 임곗값을 적용
custom_threshold = 0.5

# predict_proba( ) 반환값의 두번째 컬럼 , 즉 Positive 클래스 컬럼 하나만 추출하여 Binarizer를 적용
# - pred_proba[:, 1] : P(class=1) 1차원 ndarray
# - .reshape(-1,1)   : Binarizer는 2D 배열을 받으므로 (n,1)로 변환
pred_proba_1 = pred_proba[:,1].reshape(-1,1)

# [작동 방식]
# - Binarizer.fit는 사실상 학습할 게 없지만 일관된 인터페이스를 위해 호출
# - transform이 0.5 기준 0/1로 이진화
# - 이 결과는 lr_clf.predict()와 정확히 동일 → 임곗값 0.5의 의미 확인
binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_1) 
custom_predict = binarizer.transform(pred_proba_1)

get_clf_eval(y_test, custom_predict)

**분류 결정 임계값 0.4 기반에서 Binarizer를 이용하여 예측값 변환**

In [ ]:
# Binarizer의 threshold 설정값을 0.4로 설정. 즉 분류 결정 임곗값을 0.5에서 0.4로 낮춤  
# [원리] 임곗값을 낮추면(0.5 → 0.4)
# - "양성으로 분류되는 기준"이 완화 → Positive로 분류되는 샘플 수↑
# - 결과적으로:
#   * TP↑, FP↑   → 재현율(Recall)↑, 정밀도(Precision)↓ 경향
#   * 즉, "놓치지 않는 능력"은 좋아지지만 "허위 경보"가 늘어남
# - 정밀도-재현율은 본질적으로 트레이드오프 관계
custom_threshold = 0.4
pred_proba_1 = pred_proba[:,1].reshape(-1,1)
binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_1) 
custom_predict = binarizer.transform(pred_proba_1)

# [작동 방식]
# - 0.4 초과인 P(1) 모두 양성으로 변환됨
# - 결과 비교(0.5 → 0.4):
#   * 재현율: 0.7705 → 0.8197 (개선)
#   * 정밀도: 0.8246 → 0.7042 (악화)
get_clf_eval(y_test , custom_predict)

**여러개의 분류 결정 임곗값을 변경하면서  Binarizer를 이용하여 예측값 변환**

In [ ]:
# 테스트를 수행할 모든 임곗값을 리스트 객체로 저장. 
# [원리] 여러 임곗값에 대해 지표 변화를 한 번에 비교
# - 임곗값을 낮출수록 → Recall↑ / Precision↓
# - 임곗값을 높일수록 → Precision↑ / Recall↓
# - 비즈니스 목표(예: 암 진단은 재현율 우선, 스팸 필터는 정밀도 우선)에 맞춰 선택
thresholds = [0.4, 0.45, 0.50, 0.55, 0.60]

# [작동 방식]
# - thresholds 리스트를 순회하며 각 임곗값마다:
#   1) Binarizer로 양성 확률을 0/1 로 이진화
#   2) get_clf_eval로 오차행렬/정확도/정밀도/재현율 출력
# - 임곗값이 커질수록 정밀도가 단조 증가, 재현율이 단조 감소하는 경향을 시각적으로 확인 가능
def get_eval_by_threshold(y_test , pred_proba_c1, thresholds):
    # thresholds list객체내의 값을 차례로 iteration하면서 Evaluation 수행.
    for custom_threshold in thresholds:
        binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_c1) 
        custom_predict = binarizer.transform(pred_proba_c1)
        print('임곗값:',custom_threshold)
        get_clf_eval(y_test , custom_predict)

get_eval_by_threshold(y_test ,pred_proba[:,1].reshape(-1,1), thresholds )

**precision_recall_curve( ) 를 이용하여 임곗값에 따른 정밀도-재현율 값 추출**

In [ ]:
from sklearn.metrics import precision_recall_curve

# 레이블 값이 1일때의 예측 확률을 추출 
# [원리] precision_recall_curve(y_true, y_proba)
# - 양성 확률을 정렬하면서, 각 확률 값을 임곗값(threshold)으로 사용했을 때의
#   (precision, recall) 쌍을 한 번에 계산해 반환
# - 반환값 :
#   * precisions : 각 임곗값에서의 정밀도 ndarray (마지막에 1.0이 추가되어 길이 +1)
#   * recalls    : 각 임곗값에서의 재현율 ndarray (마지막에 0.0이 추가)
#   * thresholds : 사용된 임곗값들의 ndarray
#   → precisions/recalls 길이 = thresholds 길이 + 1
pred_proba_class1 = lr_clf.predict_proba(X_test)[:, 1] 

# 실제값 데이터 셋과 레이블 값이 1일 때의 예측 확률을 precision_recall_curve 인자로 입력 
precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_class1 )
print('반환된 분류 결정 임곗값 배열의 Shape:', thresholds.shape)
print('반환된 precisions 배열의 Shape:', precisions.shape)
print('반환된 recalls 배열의 Shape:', recalls.shape)

print("thresholds 5 sample:", thresholds[:5])
print("precisions 5 sample:", precisions[:5])
print("recalls 5 sample:", recalls[:5])

#반환된 임계값 배열 로우가 147건이므로 샘플로 10건만 추출하되, 임곗값을 15 Step으로 추출. 
# [작동 방식]
# - np.arange(0, N, 15) : 0, 15, 30, ... 형태의 인덱스를 생성
# - thr_index 위치에 해당하는 임곗값/정밀도/재현율 만 추출하여 가독성 있게 출력
# - 임곗값이 커질수록 precision↑, recall↓ 인 패턴이 확연히 드러남
thr_index = np.arange(0, thresholds.shape[0], 15)
print('샘플 추출을 위한 임계값 배열의 index 10개:', thr_index)
print('샘플용 10개의 임곗값: ', np.round(thresholds[thr_index], 2))

# 15 step 단위로 추출된 임계값에 따른 정밀도와 재현율 값 
print('샘플 임계값별 정밀도: ', np.round(precisions[thr_index], 3))
print('샘플 임계값별 재현율: ', np.round(recalls[thr_index], 3))

**임곗값의 변경에 따른 정밀도-재현율 변화 곡선을 그림**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

# [원리] 임곗값(threshold)에 따른 정밀도/재현율 변화를 한 화면에 시각화
# - x축 : threshold
# - y축 : precision(점선), recall(실선)
# - 두 곡선이 교차하는 지점 = 정밀도와 재현율의 균형점(F1이 최대화되기 쉬운 지점)
def precision_recall_curve_plot(y_test , pred_proba_c1):
    # threshold ndarray와 이 threshold에 따른 정밀도, 재현율 ndarray 추출. 
    precisions, recalls, thresholds = precision_recall_curve( y_test, pred_proba_c1)
    
    # X축을 threshold값으로, Y축은 정밀도, 재현율 값으로 각각 Plot 수행. 정밀도는 점선으로 표시
    # [작동 방식]
    # - precisions/recalls 길이는 thresholds 길이보다 1 더 큼
    #   → thresholds 길이만큼만 잘라(슬라이싱) x와 길이를 맞춤
    plt.figure(figsize=(8,6))
    threshold_boundary = thresholds.shape[0]
    plt.plot(thresholds, precisions[0:threshold_boundary], linestyle='--', label='precision')
    plt.plot(thresholds, recalls[0:threshold_boundary],label='recall')
    
    # threshold 값 X 축의 Scale을 0.1 단위로 변경
    # - plt.xlim() : 자동 설정된 (start, end) 가져오기
    # - plt.xticks(...): 0.1 간격으로 눈금 재설정 → 가독성 향상
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1),2))
    
    # x축, y축 label과 legend, 그리고 grid 설정
    plt.xlabel('Threshold value'); plt.ylabel('Precision and Recall value')
    plt.legend(); plt.grid()
    plt.show()
    
precision_recall_curve_plot( y_test, lr_clf.predict_proba(X_test)[:, 1] )


### 3.4 F1 Score

In [ ]:
from sklearn.metrics import f1_score 

# [원리] F1 Score
# - 정의: F1 = 2 * (Precision * Recall) / (Precision + Recall)
#         = 정밀도와 재현율의 조화 평균(harmonic mean)
# - 산술 평균 대신 조화 평균을 쓰는 이유:
#   둘 중 한 값이 0에 가까우면 결과도 강하게 0으로 끌려감
#   → "한쪽으로 치우친 모델"에 패널티를 주어 균형을 강조
# - 정밀도와 재현율이 비슷할수록 F1이 커지고, 둘이 일치할 때 최댓값에 가까워짐
#
# [작동 방식]
# - 임곗값 0.5 기반 pred(셀 19에서 만들어둔 변수)와 y_test로 F1 계산
f1 = f1_score(y_test , pred)
print('F1 스코어: {0:.4f}'.format(f1))


In [ ]:
# [원리] 평가 함수에 F1 스코어를 추가하여 임곗값별 모델 평가에 균형 지표를 함께 사용
# - 정확도/정밀도/재현율 만으론 트레이드오프가 가려져 보이지 않음
# - F1을 같이 보면 어떤 임곗값이 "정밀도-재현율 균형"이 가장 좋은지 한눈에 비교 가능
def get_clf_eval(y_test , pred):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    # F1 스코어 추가
    f1 = f1_score(y_test,pred)
    print('오차 행렬')
    print(confusion)
    # f1 score print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}, F1:{3:.4f}'.format(accuracy, precision, recall, f1))

# [작동 방식]
# - 5가지 임곗값(0.4~0.6)에 대해 get_eval_by_threshold를 호출
# - 각 임곗값별 정확도/정밀도/재현율/F1을 비교
# - F1이 가장 높은 임곗값이 균형이 가장 좋은 운영점(operating point)
thresholds = [0.4 , 0.45 , 0.50 , 0.55 , 0.60]
pred_proba = lr_clf.predict_proba(X_test)
get_eval_by_threshold(y_test, pred_proba[:,1].reshape(-1,1), thresholds)


## 3-5 ROC Curve와 AUC

In [ ]:
from sklearn.metrics import roc_curve

# [원리] ROC Curve (Receiver Operating Characteristic Curve)
# - 임곗값을 1에서 0으로 점차 낮추면서 (FPR, TPR) 좌표를 따라 그린 곡선
#   * TPR(True Positive Rate, 재현율) = TP / (TP + FN)
#   * FPR(False Positive Rate)        = FP / (FP + TN) = 1 - 특이도(Specificity)
# - 좌상단(0,1)에 가까울수록 좋은 모델
#   * 좌상단 = "FPR은 0인데 TPR은 1" → 모든 양성을 잡으면서 거짓 경보 0
# - 대각선(y=x)은 무작위 분류기의 성능선

# 레이블 값이 1일때의 예측 확률을 추출 
pred_proba_class1 = lr_clf.predict_proba(X_test)[:, 1] 

# [작동 방식] roc_curve(y_true, y_score)
# - y_score(여기선 양성 확률)를 내림차순으로 정렬하면서 임곗값을 변경
# - 각 임곗값에서의 (FPR, TPR)을 계산해 ndarray로 반환
# - thresholds[0]는 max(score)+1로 임의 설정됨(곡선의 시작점 (0,0)을 보장하기 위한 관례)
fprs , tprs , thresholds = roc_curve(y_test, pred_proba_class1)
# 반환된 임곗값 배열에서 샘플로 데이터를 추출하되, 임곗값을 5 Step으로 추출. 
# thresholds[0]은 max(예측확률)+1로 임의 설정됨. 이를 제외하기 위해 np.arange는 1부터 시작
thr_index = np.arange(1, thresholds.shape[0], 5)
print('샘플 추출을 위한 임곗값 배열의 index:', thr_index)
print('샘플 index로 추출한 임곗값: ', np.round(thresholds[thr_index], 2))

# 5 step 단위로 추출된 임계값에 따른 FPR, TPR 값
# - 임곗값이 낮아질수록 FPR과 TPR이 동시에 증가하는 추세
# - 좋은 모델은 같은 FPR에서 더 높은 TPR을 달성
print('샘플 임곗값별 FPR: ', np.round(fprs[thr_index], 3))
print('샘플 임곗값별 TPR: ', np.round(tprs[thr_index], 3))


In [ ]:
# [원리] ROC Curve 시각화 함수
# - x축: FPR (1 - Specificity)
# - y축: TPR (Recall, 민감도)
# - 곡선이 좌상단에 붙을수록 좋은 분류기
# - k--로 그려진 대각선은 "Random(랜덤 분류기)" 기준선 → 곡선이 이 위쪽에 있어야 의미 있음
def roc_curve_plot(y_test , pred_proba_c1):
    # 임곗값에 따른 FPR, TPR 값을 반환 받음. 
    # [작동 방식] roc_curve가 (FPR, TPR, thresholds)를 반환
    fprs , tprs , thresholds = roc_curve(y_test ,pred_proba_c1)

    # ROC Curve를 plot 곡선으로 그림. 
    plt.plot(fprs , tprs, label='ROC')
    # 가운데 대각선 직선을 그림. 
    # - (0,0)→(1,1) 직선: 무작위 추측의 성능 한계선
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    
    # FPR X 축의 Scale을 0.1 단위로 변경, X,Y 축명 설정등   
    # - xticks: 0.1 간격으로 눈금 설정 → 가독성 향상
    # - xlim/ylim: ROC는 [0,1]x[0,1] 범위에서 정의되므로 명시적으로 고정
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1),2))
    plt.xlim(0,1); plt.ylim(0,1)
    plt.xlabel('FPR( 1 - Specificity )'); plt.ylabel('TPR( Recall )')
    plt.legend()
    plt.show()
    
roc_curve_plot(y_test, lr_clf.predict_proba(X_test)[:, 1] )


In [ ]:
from sklearn.metrics import roc_auc_score

# [원리] AUC (Area Under the ROC Curve)
# - ROC 곡선 아래의 면적(0~1)을 하나의 스칼라 값으로 요약한 지표
# - 의미적 해석: "임의의 양성 샘플이 임의의 음성 샘플보다 더 높은 점수를 받을 확률"
# - 기준점:
#   * AUC = 0.5 → 무작위 분류기 수준
#   * AUC = 1.0 → 완벽한 분류기
#   * AUC > 0.9 → 매우 우수, 0.8~0.9 → 우수, 0.7~0.8 → 보통
# - 클래스 불균형에 비교적 강건하여 정확도 대안 지표로 자주 사용
#
# [작동 방식]
# - roc_auc_score(y_true, y_score)는 양성 클래스의 "확률 또는 점수"를 받음
#   → 클래스 라벨이 아니라 확률을 넣어야 함에 주의
pred_proba = lr_clf.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test, pred_proba)
print('ROC AUC 값: {0:.4f}'.format(roc_score))


In [ ]:
# [원리] 평가 지표를 종합한 최종 함수
# - 정확도/정밀도/재현율/F1 + ROC-AUC 까지 한 번에 계산 및 출력
# - pred(임곗값 적용된 0/1 라벨)와 pred_proba(양성 확률)를 모두 인자로 받는 이유:
#   * 정확도/정밀도/재현율/F1: 0/1 예측 라벨이 필요
#   * ROC-AUC: 양성 클래스의 확률(또는 점수)이 필요
#   → 두 종류의 입력을 모두 받아 한 번의 호출로 모든 지표를 산출
#
# [작동 방식]
# 1) confusion_matrix로 TN/FP/FN/TP 분포 계산
# 2) 정확도/정밀도/재현율/F1 → 라벨 기반 계산
# 3) roc_auc_score → 확률 기반 계산
# 4) 결과를 한 번에 print
def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
          F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))
